# Step 2: train YOLO26-seg with one filtered pseudo-labelling round (Colab)

**Training data:** `ScrewAndBolt_20240713` only. `Screws_2024_07_15` is not in the dataset.

1. **Round 1:** train on the consensus labels (SAM 2.1 masks reconciled with Non-AI v2 objects).
2. **Pseudo-labelling:** the round 1 model repairs labels on the *training tiles only*, under strict rules (`pseudo_label_round.py`).
3. **Round 2:** train again from the same pretrained weights with the same settings, so only the labels differ.
4. **Test:** score both models on the 5 held-out images.

**Runtime:** choose **T4 GPU**, then **Run all**, then upload `AI/datasets/challenge_seg.zip` when asked. Allow about 1 hour. Round 1 results download before round 2 starts, in case the session disconnects.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics==8.4.202
import ultralytics; ultralytics.checks()

In [ ]:
import os, glob, json, zipfile, yaml
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import files
from ultralytics import YOLO
os.chdir('/content')
if not os.path.exists('/content/challenge_seg/data.yaml'):
    up = files.upload()                                   # challenge_seg.zip
    zipfile.ZipFile(next(iter(up))).extractall('/content/')
DS, DS2 = '/content/challenge_seg', '/content/challenge_seg_r2'
split = json.load(open(f'{DS}/split.json'))
print({k: len(v) for k, v in split.items()})
assert not any('Screws_2024' in i for v in split.values() for i in v), 'Screws folder must not be in this dataset'
assert not set(split['train']) & set(split['test']), 'train/test overlap'

def write_yaml(ds):
    cfg = yaml.safe_load(open(f'{DS}/data.yaml')); cfg['path'] = ds
    yaml.safe_dump(cfg, open(f'{ds}/data_colab.yaml', 'w'))
write_yaml(DS)

### Sanity check: labels on a few training tiles

In [ ]:
tiles = sorted(glob.glob(f'{DS}/images/train/*.jpg'))
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, p in zip(ax, tiles[::max(1, len(tiles)//4)][:4]):
    img = cv2.imread(p); h, w = img.shape[:2]
    for line in open(p.replace('/images/', '/labels/').replace('.jpg', '.txt')):
        v = np.array(line.split()[1:], float).reshape(-1, 2) * [w, h]
        cv2.polylines(img, [v.astype(np.int32)], True, (0, 0, 255), 2)
    a.imshow(img[..., ::-1]); a.axis('off')

## Scripts (identical to `AI/src/`)

In [ ]:
%%writefile count_ai.py
"""
Count objects and draw masks with the fine-tuned YOLO26-seg model.

Per image
---------
1. Resize so the long side is 2048 px (the same scale as training).
2. Sliced inference: 640x640 tiles with 160 px overlap.
3. YOLO26-seg on each tile -> instance masks + confidences.
4. De-duplication: detections near a tile's border are dropped (the
   neighbouring tile sees them whole), then a global NMS in full-image
   coordinates removes overlapping and truncated (contained) duplicates.
5. Count = number of kept instances. Each instance's mask is pasted into a
   full-size label image for the overlay.
6. Writes overlays and predictions.csv (same format as the Non-AI solution).

Usage:
    python src/count_ai.py --weights weights/best.pt --data <dataset_root> --out results
    python src/count_ai.py --weights weights/best.pt --images <folder of jpgs> --out results
"""
import argparse
import csv
import glob
import os

import cv2
import numpy as np
from ultralytics import YOLO

WORK_SIZE = 2048
TILE, OVERLAP = 640, 160


def load(path, work=WORK_SIZE):
    img = cv2.imread(path)
    s = min(1.0, work / max(img.shape[:2]))
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img


def origins(n):
    if n <= TILE:
        return [0]
    return list(range(0, n - TILE, TILE - OVERLAP)) + [n - TILE]


def owned(x0, n, size):
    m = OVERLAP // 2
    return (0 if x0 == 0 else m), (size if x0 + size >= n else size - m)


def global_nms(boxes, scores, iou_thr=0.5, contain_thr=0.7):
    """Suppress duplicates across tiles in full-image coordinates.

    A box is a duplicate if it overlaps a higher-scoring box with IoU > iou_thr,
    or if most of it (> contain_thr of its area) lies inside a kept box. The
    second rule removes truncated copies of an object cut by a tile edge.
    """
    order = np.argsort(-scores)
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    keep = []
    for i in order:
        ok = True
        for j in keep:
            ix = max(0, min(boxes[i, 2], boxes[j, 2]) - max(boxes[i, 0], boxes[j, 0]))
            iy = max(0, min(boxes[i, 3], boxes[j, 3]) - max(boxes[i, 1], boxes[j, 1]))
            inter = ix * iy
            if inter == 0:
                continue
            if inter / (area[i] + area[j] - inter) > iou_thr or inter / area[i] > contain_thr:
                ok = False
                break
        if ok:
            keep.append(i)
    return keep


def count_image(model, img, conf=0.25, device=None):
    """Sliced inference -> global de-duplication -> (count, label image)."""
    H, W = img.shape[:2]
    boxes, scores, masks = [], [], []
    for y0 in origins(H):
        for x0 in origins(W):
            t = img[y0:y0 + TILE, x0:x0 + TILE]
            th, tw = t.shape[:2]
            r = model.predict(t, imgsz=TILE, conf=conf, max_det=1000, retina_masks=True,
                              verbose=False, device=device)[0]
            if r.boxes is None or len(r.boxes) == 0:
                continue
            xy = r.boxes.xyxy.cpu().numpy()
            sc = r.boxes.conf.cpu().numpy()
            ms = r.masks.data.cpu().numpy() > 0.5 if r.masks is not None else None
            # pre-filter: drop detections whose centre is outside this tile's
            # central region (they are seen better by the neighbouring tile)
            ylo, yhi = owned(y0, H, th)
            xlo, xhi = owned(x0, W, tw)
            for k, b in enumerate(xy):
                cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
                if not (xlo - OVERLAP // 4 <= cx < xhi + OVERLAP // 4 and
                        ylo - OVERLAP // 4 <= cy < yhi + OVERLAP // 4):
                    continue
                bx0, by0, bx1, by1 = b.astype(int)
                boxes.append([b[0] + x0, b[1] + y0, b[2] + x0, b[3] + y0])
                scores.append(sc[k])
                m = ms[k][by0:by1, bx0:bx1] if ms is not None else None
                masks.append((x0 + bx0, y0 + by0, m))
    seg = np.zeros((H, W), np.int32)
    if not boxes:
        return 0, seg
    keep = global_nms(np.array(boxes), np.array(scores))
    for n, i in enumerate(sorted(keep, key=lambda i: -scores[i]), 1):
        gx, gy, m = masks[i]
        if m is None or m.size == 0:
            continue
        region = seg[gy:gy + m.shape[0], gx:gx + m.shape[1]]
        m = m[:region.shape[0], :region.shape[1]]
        region[m & (region == 0)] = n
    return len(keep), seg


def overlay(img, seg, count):
    rng = np.random.default_rng(0)
    colors = rng.integers(60, 255, size=(seg.max() + 1, 3)).astype(np.uint8)
    colors[0] = 0
    out = img.copy()
    fg = seg > 0
    out[fg] = (0.45 * img[fg] + 0.55 * colors[seg][fg]).astype(np.uint8)
    edges = cv2.morphologyEx(fg.astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8))
    out[edges > 0] = (0, 0, 255)
    txt = f"Count: {count}"
    cv2.rectangle(out, (10, 10), (20 + 28 * len(txt), 75), (0, 0, 0), -1)
    cv2.putText(out, txt, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (255, 255, 255), 3)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data", help="dataset root (folders of .jpg)")
    ap.add_argument("--images", help="a single folder of .jpg")
    ap.add_argument("--out", default="results")
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--device", default=None)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    model = YOLO(args.weights)
    if args.data:
        files = sorted(glob.glob(os.path.join(args.data, "*", "*.jpg")))
        names = [f"{os.path.basename(os.path.dirname(f))}/{os.path.basename(f)}" for f in files]
    else:
        files = sorted(glob.glob(os.path.join(args.images, "*.jpg")))
        names = [os.path.basename(f)[:-4].replace("__", "/") + ".jpg" for f in files]
    rows = []
    for f, name in zip(files, names):
        img = load(f)
        n, seg = count_image(model, img, args.conf, args.device)
        cv2.imwrite(os.path.join(args.out, name.replace("/", "__")), overlay(img, seg, n))
        rows.append({"image": name, "predicted": n})
        print(f"{name:45s} count={n:4d}", flush=True)
    with open(os.path.join(args.out, "predictions.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["image", "predicted"])
        w.writeheader()
        w.writerows(rows)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile pseudo_label_round.py
"""
One filtered pseudo-labelling round on the TRAINING tiles.

A trained YOLO-seg model predicts on every training tile, and its predictions
are used to improve the existing labels. Raw predictions are never accepted as
they are, because a model would then reinforce its own mistakes.

Rules, applied per tile:
  KEEP     every existing label stays. Nothing is ever deleted.
  REFINE   a prediction replaces a label's outline only if conf >= 0.70,
           IoU >= 0.60 and its area is 1.10-1.40x the label's area. This
           repairs labels that are too small (e.g. a partly masked shank)
           and leaves good SAM outlines alone.
  MERGE    an unmatched prediction (conf >= 0.70) that touches a label and is
           smaller than 0.6x the median label area is a missing part, such as
           a shank beside a labelled head. It is joined to that label.
  ADD      an unmatched prediction becomes a new label only if conf >= 0.70,
           it touches no label, it does not touch the tile border, and it is
           at least 0.25x the median label area.
  SKIP     anything else, including large predictions touching a label,
           because those cases are ambiguous.

Only images/train is changed. Validation tiles and test images are not read,
so the held-out test stays independent of this round.

Usage:
    python src/pseudo_label_round.py --weights best.pt --dataset datasets/challenge_seg \
        --out datasets/challenge_seg_r2 --device 0
"""
import argparse
import glob
import json
import os
import shutil

import cv2
import numpy as np
from ultralytics import YOLO

REFINE_CONF, REFINE_IOU, REFINE_RATIO = 0.70, 0.60, (1.10, 1.40)
ADD_CONF, ADD_MIN_AREA, MERGE_MAX_AREA, BORDER, TOUCH_PX = 0.70, 0.25, 0.60, 3, 5


def read_labels(path, w, h):
    polys = []
    if os.path.exists(path):
        for line in open(path):
            v = line.split()[1:]
            if len(v) >= 6:
                polys.append((np.array(v, float).reshape(-1, 2) * [w, h]).astype(np.float32))
    return polys


def raster(poly, shape):
    m = np.zeros(shape, np.uint8)
    cv2.fillPoly(m, [poly.astype(np.int32)], 1)
    return m.astype(bool)


def to_line(poly, w, h):
    p = np.clip(poly / [w, h], 0, 1)
    return "0 " + " ".join(f"{v:.5f}" for v in p.reshape(-1))


def merge_tile(labels, preds, confs, shape, min_area):
    """Return (polygons, n_refined, n_added, n_merged) for one tile."""
    h, w = shape
    lab_m = [raster(p, shape) for p in labels]
    prd_m = [raster(p, shape) for p in preds]
    out = list(labels)
    pairs = []
    for j, pm in enumerate(prd_m):
        for i, lm in enumerate(lab_m):
            inter = np.logical_and(pm, lm).sum()
            if inter:
                pairs.append((inter / np.logical_or(pm, lm).sum(), i, j))
    used_l, used_p, refined = set(), set(), 0
    for iou, i, j in sorted(pairs, reverse=True):        # one-to-one matching
        if iou < REFINE_IOU or i in used_l or j in used_p:
            continue
        used_l.add(i)
        used_p.add(j)
        ratio = prd_m[j].sum() / max(1, lab_m[i].sum())
        if confs[j] >= REFINE_CONF and REFINE_RATIO[0] <= ratio <= REFINE_RATIO[1]:
            out[i] = preds[j]
            refined += 1
    kernel = np.ones((2 * TOUCH_PX + 1, 2 * TOUCH_PX + 1), np.uint8)
    added = merged = 0
    for j, pm in enumerate(prd_m):
        a = pm.sum()
        if j in used_p or confs[j] < ADD_CONF or a < ADD_MIN_AREA * min_area:
            continue
        near = cv2.dilate(pm.astype(np.uint8), kernel) > 0
        contact = [np.logical_and(near, raster(p, shape)).sum() for p in out]
        if contact and max(contact) > 0:
            if a < MERGE_MAX_AREA * min_area:
                i = int(np.argmax(contact))
                joined = cv2.dilate((raster(out[i], shape) | pm).astype(np.uint8), kernel)
                joined = cv2.erode(joined, kernel)
                cnts, _ = cv2.findContours(joined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                out[i] = max(cnts, key=cv2.contourArea).reshape(-1, 2).astype(np.float32)
                merged += 1
            continue
        x, y = preds[j][:, 0], preds[j][:, 1]
        if x.min() < BORDER or y.min() < BORDER or x.max() > w - BORDER or y.max() > h - BORDER:
            continue
        out.append(preds[j])
        added += 1
    return out, refined, added, merged


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--device", default=None)
    ap.add_argument("--limit", type=int, help="process only N tiles (for testing)")
    args = ap.parse_args()
    if os.path.exists(args.out):
        shutil.rmtree(args.out)
    shutil.copytree(args.dataset, args.out)

    tiles = sorted(glob.glob(os.path.join(args.dataset, "images", "train", "*.jpg")))
    if args.limit:
        tiles = tiles[::max(1, len(tiles) // args.limit)][:args.limit]
    areas = []
    for t in tiles:
        h, w = cv2.imread(t).shape[:2]
        lab = t.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)[:-4] + ".txt"
        areas += [cv2.contourArea(p) for p in read_labels(lab, w, h)]
    min_area = float(np.median(areas)) if areas else 0.0

    model = YOLO(args.weights)
    totals = {"tiles": 0, "labels_before": 0, "labels_after": 0, "refined": 0, "added": 0, "merged": 0,
              "added_by_tile": {}}
    for t in tiles:
        img = cv2.imread(t)
        h, w = img.shape[:2]
        stem = os.path.basename(t)[:-4]
        labels = read_labels(os.path.join(args.dataset, "labels", "train", stem + ".txt"), w, h)
        r = model.predict(img, imgsz=640, conf=0.25, max_det=1000, retina_masks=True,
                          verbose=False, device=args.device)[0]
        preds, confs = [], []
        if r.masks is not None:
            for p, c in zip(r.masks.xy, r.boxes.conf.cpu().numpy()):
                if len(p) >= 3:
                    preds.append(np.asarray(p, np.float32))
                    confs.append(float(c))
        result, refined, added, joined = merge_tile(labels, preds, confs, (h, w), min_area)
        with open(os.path.join(args.out, "labels", "train", stem + ".txt"), "w") as fh:
            fh.write("\n".join(to_line(p, w, h) for p in result) + ("\n" if result else ""))
        totals["tiles"] += 1
        totals["labels_before"] += len(labels)
        totals["labels_after"] += len(result)
        totals["refined"] += refined
        totals["added"] += added
        totals["merged"] += joined
        if added:
            totals["added_by_tile"][stem] = added
    json.dump(totals, open(os.path.join(args.out, "pseudo_label_report.json"), "w"), indent=1)
    print({k: v for k, v in totals.items() if k != "added_by_tile"})


if __name__ == "__main__":
    main()


## Round 1: train on the consensus labels

In [ ]:
model = YOLO('yolo26s-seg.pt')
model.train(data=f'{DS}/data_colab.yaml', imgsz=640, epochs=120, batch=16, device=0,
            degrees=180, flipud=0.5, fliplr=0.5, scale=0.3, mosaic=1.0, close_mosaic=15,
            patience=30, project='/content/runs', name='round1', exist_ok=True,
            workers=2, plots=True)

In [ ]:
R1 = '/content/runs/round1/weights/best.pt'
!python count_ai.py --weights {R1} --images {DS}/test_full --out test_round1 --device 0
!zip -qr round1.zip runs/round1 test_round1
files.download('round1.zip')

## Pseudo-labelling round (training tiles only)
Every label is kept. Predictions can only enlarge a too-small label, join a missing part to the label it touches, or add an isolated object. Validation tiles and test images are not read.

In [ ]:
!python pseudo_label_round.py --weights {R1} --dataset {DS} --out {DS2} --device 0
report = json.load(open(f'{DS2}/pseudo_label_report.json'))
print({k: v for k, v in report.items() if k != 'added_by_tile'})
write_yaml(DS2)

### Review the isolated objects that were added
Green = labels, red = added. If the red areas are not real objects, stop here and use round 1.

In [ ]:
added = list(report['added_by_tile'])[:8]
print(len(report['added_by_tile']), 'tiles with added objects')
if added:
    fig, ax = plt.subplots(2, 4, figsize=(20, 10))
    for a in ax.ravel(): a.axis('off')
    for a, stem in zip(ax.ravel(), added):
        img = cv2.imread(f'{DS2}/images/train/{stem}.jpg'); h, w = img.shape[:2]
        rows = open(f'{DS2}/labels/train/{stem}.txt').read().splitlines()
        n_old = len(open(f'{DS}/labels/train/{stem}.txt').read().splitlines())
        for k, line in enumerate(rows):
            v = (np.array(line.split()[1:], float).reshape(-1, 2) * [w, h]).astype(np.int32)
            cv2.polylines(img, [v], True, (0, 255, 0) if k < n_old else (0, 0, 255), 2 if k < n_old else 5)
        a.imshow(img[..., ::-1]); a.set_title(stem[-20:])

## Round 2: train on the improved labels

In [ ]:
model = YOLO('yolo26s-seg.pt')
model.train(data=f'{DS2}/data_colab.yaml', imgsz=640, epochs=120, batch=16, device=0,
            degrees=180, flipud=0.5, fliplr=0.5, scale=0.3, mosaic=1.0, close_mosaic=15,
            patience=30, project='/content/runs', name='round2', exist_ok=True,
            workers=2, plots=True)

## Compare the two rounds
Choose the model using the **validation score**, which both rounds measure on the same unchanged tiles. Do not choose using the test table: it must stay an unbiased final score.

Test counts are compared with the consensus labels, whose counts come from Non-AI v2, so this measures agreement with those labels rather than verified truth.

In [ ]:
R2 = '/content/runs/round2/weights/best.pt'
!python count_ai.py --weights {R2} --images {DS}/test_full --out test_round2 --device 0
val = {}
for name, w in (('round1', R1), ('round2', R2)):
    m = YOLO(w).val(data=f'{DS}/data_colab.yaml', imgsz=640, max_det=1000, verbose=False)
    val[name] = {'box_mAP50': m.box.map50, 'mask_mAP50': m.seg.map50, 'mask_mAP50-95': m.seg.map}
print(pd.DataFrame(val).T.round(3))

df = pd.read_csv(f'{DS}/test_full_labels.csv')
for name in ('round1', 'round2'):
    p = pd.read_csv(f'test_{name}/predictions.csv').rename(columns={'predicted': name})
    df = df.merge(p, on='image')
    df[name + '_acc%'] = (1 - (df[name] - df.label_count).abs() / df.label_count) * 100
print(df.round(1).to_string(index=False))
print(df[['round1_acc%', 'round2_acc%']].mean().round(2))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(21, 13))
for a in ax.ravel(): a.axis('off')
for a, p in zip(ax.ravel(), sorted(glob.glob('test_round2/*.jpg'))):
    a.imshow(cv2.imread(p)[..., ::-1]); a.set_title(os.path.basename(p))

## Download
`round1.zip` downloaded earlier. Use the weights of the round chosen by validation score.

In [ ]:
!cp {DS2}/pseudo_label_report.json runs/round2/
!zip -qr round2.zip runs/round2 test_round2
files.download('round2.zip')